# EcoSort Waste Management Assistant- PART 2


In [ ]:
# Apply the preprocessing function to the datasets

print("Training batches:",
      tf.data.experimental.cardinality(train_ds).numpy())

print("Validation batches:",
      tf.data.experimental.cardinality(validation_ds).numpy())

print("Test batches:",
      tf.data.experimental.cardinality(test_dataset).numpy())

# Retrieving one batch from the training dataset
sample_images, sample_labels = next(iter(train_ds))

print("\nImage Batch Shape:", sample_images.shape)
print("Label Batch Shape:", sample_labels.shape)

# Displaying a sample of preprocessed images
plt.figure(figsize=(12, 8))

for i in range(min(9, len(sample_images))):

    plt.subplot(3, 3, i + 1)

    plt.imshow(
        tf.cast(
            tf.clip_by_value(
                sample_images[i],
                0,
                255
            ),
            tf.uint8
        )
    )

    plt.title(
        class_names[
            np.argmax(sample_labels[i].numpy())
        ]
    )

    plt.axis("off")

plt.suptitle(
    "Sample Preprocessed Images",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

In [ ]:
# Image augmentation pipeline using Keras preprocessing layers

image_augmentation = keras.Sequential(
    [
        layers.RandomFlip(
            "horizontal",
            seed=SEED
        ),
        layers.RandomRotation(
            factor=0.10,
            seed=SEED
        ),
        layers.RandomZoom(
            height_factor=0.10,
            width_factor=0.10,
            seed=SEED
        ),
        layers.RandomContrast(
            factor=0.10,
            seed=SEED
        )
    ],
    name="image_augmentation"
)

In [ ]:
# Implement CNN model using transfer learning with EfficientNetB0

# Loading EfficientNetB0 without its original classifier
base_model = EfficientNetB0(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)
)

# Freezing pre-trained layers during initial training
base_model.trainable = False

# Building the transfer-learning model
cnn_model = keras.Sequential(
    [
        # Explicit input definition improves model serialization
        layers.Input(
            shape=(IMG_HEIGHT, IMG_WIDTH, 3),
            name="image_input"
        ),

        # Apply augmentation only during training
        image_augmentation,

        # EfficientNetB0 includes its required input rescaling
        base_model,

        # Convert feature maps into a feature vector
        layers.GlobalAveragePooling2D(),

        # Stabilize activations
        layers.BatchNormalization(),

        # First classification layer
        layers.Dense(
            512,
            activation="relu",
            kernel_regularizer=regularizers.l2(0.001)
        ),

        # Reduce overfitting
        layers.Dropout(0.50),

        # Second classification layer
        layers.Dense(
            256,
            activation="relu",
            kernel_regularizer=regularizers.l2(0.001)
        ),

        # Additional regularization
        layers.Dropout(0.30),

        # Nine-class output layer
        layers.Dense(
            num_classes,
            activation="softmax"
        )
    ],
    name="ecosort_cnn"
)

# Display model architecture
cnn_model.summary()

In [ ]:
# Compiling the CNN model with Adam optimizer and categorical cross-entropy loss

cnn_model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=0.001
    ),

    loss="categorical_crossentropy",

    metrics=[
        "accuracy"
    ]
)

print("CNN model compiled successfully.")


In [ ]:
# Configuring training callbacks for early stopping, learning rate reduction, and model checkpointing

early_stopping = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=3,
    min_lr=1e-6
)

checkpoint = callbacks.ModelCheckpoint(
    "best_cnn_model.keras",
    monitor="val_accuracy",
    save_best_only=True
)

print("Training callbacks created successfully.")

In [ ]:
# Checking class weights

from sklearn.utils.class_weight import compute_class_weight

image_labels = image_df["category"]

label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(image_labels)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(encoded_labels),
    y=encoded_labels
)

class_weights = {
    i: weight
    for i, weight in enumerate(class_weights)
}

print(class_weights)

In [ ]:
# Training the CNN Model
# Configuring training callbacks
early_stopping = callbacks.EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.2,
    patience=3,
    min_lr=1e-6
)

checkpoint = callbacks.ModelCheckpoint(
    "best_cnn_model.keras",
    monitor="val_accuracy",
    save_best_only=True,
    verbose=1
)

# Train the model
history = cnn_model.fit(
    train_ds,
    class_weight=class_weights,
    validation_data=validation_ds,
    epochs=15,
    callbacks=[
        early_stopping,
        reduce_lr,
        checkpoint
    ]
)

print("Model training completed.")

In [ ]:
# Save baseline CNN model for future use

cnn_model.save("baseline_cnn_model.keras")

print("Baseline CNN model saved successfully.")

In [ ]:
# Visualizing the training performance metrics (accuracy and loss) over epochs


fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy
axes[0].plot(
    history.history["accuracy"],
    label="Training Accuracy"
)

axes[0].plot(
    history.history["val_accuracy"],
    label="Validation Accuracy"
)

axes[0].set_title("Model Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()

# Loss
axes[1].plot(
    history.history["loss"],
    label="Training Loss"
)

axes[1].plot(
    history.history["val_loss"],
    label="Validation Loss"
)

axes[1].set_title("Model Loss")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Evaluating model performance

# Evaluating on the test dataset
test_loss, test_accuracy = cnn_model.evaluate(
    test_dataset,
    verbose=1
)

print(f"\nTest Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

In [ ]:
# Generating test predictions


y_true = []
y_pred = []

for images, labels in test_dataset:

    predictions = cnn_model.predict(
        images,
        verbose=0
    )

    y_true.extend(
        np.argmax(labels.numpy(), axis=1)
    )

    y_pred.extend(
        np.argmax(predictions, axis=1)
    )

y_true = np.array(y_true)
y_pred = np.array(y_pred)


In [ ]:
# Classification report

report = classification_report(
    y_true,
    y_pred,
    target_names=class_names
)

print(report)

In [ ]:
# Confusion matrix


cm = confusion_matrix(
    y_true,
    y_pred
)

plt.figure(figsize=(10, 8))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.title("CNN Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")

plt.tight_layout()
plt.show()

In [ ]:
# Identifying the most challenging categories based on accuracy

category_accuracy = (
    cm.diagonal()
    / cm.sum(axis=1)
)

category_performance = pd.DataFrame({
    "Category": class_names,
    "Accuracy": category_accuracy
})

category_performance = (
    category_performance
    .sort_values(
        "Accuracy",
        ascending=True
    )
)

display(category_performance)

print(
    "\nMost challenging categories:"
)

display(
    category_performance.head(3)
)

### Classification error analysis
Glass and Miscellaneous Trash achieved the lowest accuracies, likely due to visual similarities, transparent materials, and high within-class variation. These findings suggest the need for additional training data and targeted augmentation to improve classification performance.

#### CNN performance summary
The baseline CNN achieved 85.0% test accuracy with a test loss of 1.1762, demonstrating strong overall classification performance. While Vegetation achieved perfect classification, Miscellaneous Trash (59.62%) and Glass (62.16%) were the most challenging classes due to overlapping visual characteristics and higher intra-class variability.

In [ ]:
# Fine tuning the model

# Unfreezing the EfficientNet base model
base_model.trainable = True

# Freezing lower layers and fine-tune upper layers
fine_tune_at = 200

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

print(f"Total layers in EfficientNetB0: {len(base_model.layers)}")
print(f"Layers frozen: {fine_tune_at}")
print(
    f"Layers trainable: "
    f"{len(base_model.layers) - fine_tune_at}"
)

# Recompiling using a much smaller learning rate
cnn_model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("Fine-tuning model compiled successfully.")

In [ ]:
# Training the fine-tuned model with early stopping, learning rate reduction, and checkpointing

fine_tune_history = cnn_model.fit(
    train_ds,
    validation_data=validation_ds,
    epochs=10,
    callbacks=[
        early_stopping,
        reduce_lr,
        checkpoint
    ]
)

print("Fine-tuning completed.")

In [ ]:
# Evaluating the fine-tuned model

fine_tune_loss, fine_tune_accuracy = cnn_model.evaluate(
    test_dataset,
    verbose=1
)

print(f"\nFine-Tuned Test Loss: {fine_tune_loss:.4f}")
print(f"Fine-Tuned Test Accuracy: {fine_tune_accuracy:.4f}")

In [ ]:
#CNN model deployment
import keras

# Use the available model
deployment_cnn_model = cnn_model

# Compile
deployment_cnn_model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=1e-5
    ),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("Deployment CNN model prepared successfully.")

#### Deployment model validation



In [ ]:
#Validate the selected deployment model

deployment_test_loss, deployment_test_accuracy = (
    deployment_cnn_model.evaluate(
        test_dataset,
        verbose=1
    )
)

print(
    f"Deployment Model Test Loss: "
    f"{deployment_test_loss:.4f}"
)

print(
    f"Deployment Model Test Accuracy: "
    f"{deployment_test_accuracy:.4f}"
)

assert np.isfinite(deployment_test_loss), (
    "Deployment model produced a non-finite test loss."
)

assert 0.0 <= deployment_test_accuracy <= 1.0, (
    "Deployment model accuracy is outside the expected range."
)

print(
    "Deployment model validation completed successfully."
)

In [ ]:
# Comparing baseline vs fine-tuned performance

comparison_df = pd.DataFrame({
    "Metric": ["Test Accuracy"],
    "Baseline Model": [test_accuracy],
    "Fine-Tuned Model": [fine_tune_accuracy]
})

display(comparison_df)

# Visual comparison
plt.figure(figsize=(6, 5))

sns.barplot(
    data=comparison_df.melt(
        id_vars="Metric",
        var_name="Model",
        value_name="Accuracy"
    ),
    x="Model",
    y="Accuracy",
    palette=["#0B6E4F", "#D99201"]
)

plt.title("Baseline vs Fine-Tuned CNN Performance")
plt.ylim(0, 1)

plt.show()

The baseline CNN outperformed the fine-tuned model, achieving higher test accuracy (85.00% vs. 84.17%) and lower loss (1.1762 vs. 1.1878). Consequently, the baseline model was selected for deployment, highlighting the importance of using test-set evidence to guide fine-tuning decisions.